# Sentence-score comparisons

Figure 2 and Supplementary Table S3. Run notebook 01 first.

## Step 1: Load the scores

In [ ]:
from pathlib import Path
from IPython.display import display
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'syllableseparation_apple_corrected.csv').is_file():
    raise FileNotFoundError('Open this notebook from the repository folder.')
output_dir = ROOT / 'results'
output_dir.mkdir(exist_ok=True)
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 9,
                     'axes.linewidth': .8, 'pdf.fonttype': 42})

def save_figure(fig, name):
    fig.savefig(output_dir / f'{name}.png', dpi=300, bbox_inches='tight', facecolor='white')
    fig.savefig(output_dir / f'{name}.pdf', bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close(fig)
df = pd.read_csv(ROOT / 'Syllableerror_scores_raw_and_normalized.csv', dtype={'sub_id': str})
features = [f'error_rate_{i}' for i in range(1, 5)]
raw_columns = [f'error_{i}' for i in range(1, 5)]
sentences = ['LBP', 'LAP', 'VP', 'MP']
assert len(df) == 74 and df['sub_id'].is_unique
assert df['disease'].value_counts().to_dict() == {0: 37, 1: 37}
assert np.isfinite(df[features + raw_columns].to_numpy()).all()
assert np.allclose(df[features], df[raw_columns].to_numpy() / [8, 8, 7, 11])
df['pooled'] = df[raw_columns].sum(axis=1) / 34

## Step 2: Summarize scores and compare groups

In [ ]:
from scipy import stats
from statsmodels.stats.multitest import multipletests
summary_rows, normality_rows, comparison_rows = [], [], []
for sentence, feature in zip(sentences, features):
    for label, group in [(1, 'Spastic dysarthria'), (0, 'Control')]:
        values = df.loc[df.disease == label, feature]
        q1, median, q3 = values.quantile([.25, .5, .75])
        summary_rows.append({'Sentence': sentence, 'Group': group, 'N': len(values),
                             'Median': median, 'Q1': q1, 'Q3': q3})
        w, p = stats.shapiro(values)
        normality_rows.append({'Sentence': sentence, 'Group': group, 'N': len(values),
                               'Shapiro_W': w, 'p': p})
    cases, controls = df.loc[df.disease == 1, feature], df.loc[df.disease == 0, feature]
    test = stats.mannwhitneyu(cases, controls, alternative='two-sided', method='asymptotic')
    comparison_rows.append({'Sentence': sentence, 'U_cases_first': test.statistic, 'p': test.pvalue})
summary_df = pd.DataFrame(summary_rows)
normality_df = pd.DataFrame(normality_rows)
comparison_df = pd.DataFrame(comparison_rows)
comparison_df['p_Holm'] = multipletests(comparison_df['p'], method='holm')[1]
summary_df.to_csv(output_dir / 'Figure2_score_summary.csv', index=False)
normality_df.to_csv(output_dir / 'TableS3_normality.csv', index=False)
comparison_df.to_csv(output_dir / 'Figure2_Mann_Whitney.csv', index=False)
display(summary_df.round(3))
display(normality_df.round(4))
display(comparison_df.round(6))

## Step 3: Draw Figure 2

In [ ]:
from matplotlib.patches import Patch
palette = {'Spastic dysarthria': '#E5A54C', 'Control': '#A8D4E5'}
fig, ax = plt.subplots(figsize=(180 / 25.4, 4.5))
rng = np.random.default_rng(2026)
for i, (sentence, feature) in enumerate(zip(sentences, features)):
    for label, group, offset in [(1, 'Spastic dysarthria', -.19), (0, 'Control', .19)]:
        values = df.loc[df.disease == label, feature]
        position = i + offset
        ax.boxplot(values, positions=[position], widths=.29, patch_artist=True,
                   manage_ticks=False, showfliers=False,
                   boxprops={'facecolor': palette[group], 'edgecolor': '.3'},
                   medianprops={'color': '.15', 'linewidth': 1.3})
        ax.scatter(position + rng.uniform(-.09, .09, len(values)), values,
                   s=12, facecolors='none', edgecolors='.35', linewidths=.55, alpha=.75, zorder=3)
    p = comparison_df.loc[comparison_df.Sentence == sentence, 'p_Holm'].iloc[0]
    ax.plot([i-.19, i-.19, i+.19, i+.19], [1.04, 1.058, 1.058, 1.04], color='.2', lw=.8)
    ax.text(i, 1.075, 'p < .001' if p < .001 else f'p = {p:.3f}', ha='center', va='bottom')
ax.set(xlim=(-.55, 3.55), ylim=(-.025, 1.32), xticks=range(4), xticklabels=sentences,
       yticks=np.arange(0, 1.01, .2), xlabel='Sentence', ylabel='Syllable-normalized error score')
ax.legend(handles=[Patch(facecolor=palette[g], edgecolor='.3', label=g) for g in palette],
          loc='upper center', bbox_to_anchor=(.5, .985), ncol=2, frameon=False)
fig.subplots_adjust(left=.12, right=.98, bottom=.15, top=.87)
save_figure(fig, 'Figure2_scores')